## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [3]:
## Connection
connection = presto.connect(
        host = 'bi-presto.serving.data.production.internal',
        # 'bi-trino-2.serving.data.production.internal',
        # 'bi-presto.serving.data.production.internal' ,
        #host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Methodology
### Objective: 
- To build an (a/b) experiment customer group for conducting bias-free experimentation.

### Segmentation Criteria: 
- Customer Entry Criteria (Pan-India <> Link & Auto & Cab) 
- Customer Metrics (LTR <> FE <> RR <> NET <> Segment <> COBRA <> COBRM <> OCARA <> expired)

### Collect Data: 
- fare_estimates_enriched <> order_logs_snapshot/immutable/fact <> iallocator_customer_segments
- Create a customer-level aggregated view.

### Stratified Random Sampling: 
- Use randomization to assign customers within each segment to either the control group (A) or the treatment group (B).
- <b>Balance Groups:</b>     
    - Ensure that the size of the groups is balanced. 
    - Conduct a pre-analysis to check if the groups are indeed homogeneous before starting the experiment.
    - This helps in confirming that the segmentation strategy is effective.
- <b>Exclude Outliers:</b>
    - Identify and exclude outliers that might skew the results. 
    - Outliers could be customers with extremely high or low values in the chosen criteria that are not representative of the majority.
	
### Implement the Experiment: 
- Implement the changes for the treatment group and keep the control group unchanged. 
	
### Monitor and Analyse: 
- Monitor the experiment in real-time and, once completed, analyse the results to determine if there are significant differences between the groups.

## Parameter

In [4]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/E2074/analytics/customer/Ads-Monetisation/jackport


In [5]:
local_datasets = '/Users/E2074/local-datasets/customer/ads-monetisation/jackport/'


In [6]:
## Parameter 
start_date = '2024-12-20'
end_date = '2025-01-17'
cities = ['Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Mumbai', 'Kolkata', 'Ahmedabad', 'Pune', 'Jaipur', 'Vijayawada']

services = ['Auto', 'Auto NCR', 'Auto Pet','Auto Pool','AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Link', 'CabEconomy', 'CabPremium']

In [7]:
# Convert the lists to comma-separated strings
city_str = "', '".join(city for city in cities)
service_str = "', '".join(service for service in services)

print(city_str)
print(service_str)

Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Mumbai', 'Kolkata', 'Ahmedabad', 'Pune', 'Jaipur', 'Vijayawada
Auto', 'Auto NCR', 'Auto Pet', 'Auto Pool', 'AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Link', 'CabEconomy', 'CabPremium


## Datasets

In [26]:
##customer_funnel
def func_get_customer_base(start_date, end_date, city_str, service_str):
        
    customer_funnel = f"""
    
        WITH base AS (
        
            SELECT
                customerid,
                city,
                COALESCE(SUM(ao_sessions_unique_daily), 0)ao_sessions,
                COALESCE(SUM(fe_sessions_unique_daily), 0)fe_sessions,
                COALESCE(SUM(rr_sessions_unique_daily), 0)rr_sessions,
                
                COALESCE(SUM(CASE WHEN service_name IN ('Bike Lite', 'Bike Metro', 'Bike Pink', 'Link') THEN gross_rides_daily END), 0)link_gross_rides,
                COALESCE(SUM(CASE WHEN service_name IN ('Auto', 'Auto NCR', 'Auto Pet','Auto Pool','AutoPremium') THEN gross_rides_daily END), 0)auto_gross_rides,
                COALESCE(SUM(CASE WHEN service_name IN ('CabEconomy', 'CabPremium') THEN gross_rides_daily END), 0)cab_gross_rides,
                
                COALESCE(SUM(CASE WHEN service_name IN ('Bike Lite', 'Bike Metro', 'Bike Pink', 'Link') THEN net_rides_daily END), 0)link_net_rides,
                COALESCE(SUM(CASE WHEN service_name IN ('Auto', 'Auto NCR', 'Auto Pet','Auto Pool','AutoPremium') THEN net_rides_daily END), 0) auto_net_rides,
                COALESCE(SUM(CASE WHEN service_name IN ('CabEconomy', 'CabPremium') THEN net_rides_daily END), 0) cab_net_rides
                
            FROM
                (
                SELECT 
                    day,
                    customerid,
                    city,
                    service_name,
                    MAX(ao_sessions_unique_daily) ao_sessions_unique_daily,
                    MAX(fe_sessions_unique_daily) fe_sessions_unique_daily,
                    MAX(rr_sessions_unique_daily) rr_sessions_unique_daily,
                    
                    SUM(gross_rides_daily) gross_rides_daily,
                    SUM(net_rides_daily) net_rides_daily
                    
                FROM 
                    datasets.customer_rf_daily_kpi 
                WHERE 
                    day >= '{start_date}' 
                    AND day <= '{end_date}'
                    AND city IN ('{city_str}')
                    -- ('Hyderabad', 'Bangalore', 'Delhi', 'Chennai', 'Mumbai', 'Kolkata', 'Ahmedabad', 'Pune', 'Jaipur', 'Vijayawada')
                    AND service_name IN ('{service_str}')
                    -- IN ('Auto', 'Auto NCR', 'Auto Pet','Auto Pool','AutoPremium', 'Bike Lite', 'Bike Metro', 'Bike Pink', 'Link','CabEconomy', 'CabPremium')
                GROUP BY 1,2,3,4
                )
            GROUP BY 1,2
            )

            SELECT 
                *
            FROM 
                base        
    """
    
    df_customer_funnel = pd.read_sql(customer_funnel, connection)
    print(df_customer_funnel.shape)
    df_customer_funnel.to_csv(local_datasets + 'raw/customer_data.csv',index=False)

func_get_customer_base(start_date, end_date, city_str, service_str)

(21321993, 11)


In [29]:
##customer_funnel
def func_get_customer_segment(start_date, end_date, city_str, service_str):
        
    customer_funnel = f"""
    
        WITH olf AS (
        
            SELECT
                customer_id,
                customer_obj_mobile,
                city_name
            FROM 
                (
                SELECT
                    customer_id,
                    customer_obj_mobile,
                    city_name,
                    row_number() over(partition by customer_id order by order_requested_epoch desc) rn 
                FROM 
                    orders.order_logs_fact
                WHERE
                    yyyymmdd >= '20241220'
                    AND yyyymmdd <= '20250117'
                    AND city_name IN ('{city_str}')
                    AND service_obj_service_name IN ('{service_str}')
                )
            WHERE 
                rn = 1
            ),
            
            segment AS (
            
            SELECT
                customer_id,
                gender_tag as  gender,
                taxi_retention_segments,
                taxi_lifetime_stage,
                taxi_ltr_segment,
                taxi_service_affinity,
                taxi_regularity_segment,
                taxi_fe_regularity_segment
            FROM 
                datasets.iallocator_customer_segments
            WHERE 
                run_date = '2024-12-19'
                AND fe_recency_type IN ('Recent', 'Stationary', 'Dormant')
            )
            
            SELECT
                *
            FROM 
                olf
            LEFT JOIN 
                segment
                ON olf.customer_id = segment.customer_id       
    """
    
    df_customer_funnel = pd.read_sql(customer_funnel, connection)
    print(df_customer_funnel.shape)
    df_customer_funnel.to_csv(local_datasets + 'raw/customer_segment.csv',index=False)

func_get_customer_segment(start_date, end_date, city_str, service_str)

(15604162, 11)


### Reading local extract

In [8]:
df_customer_data = pd.read_csv(local_datasets + 'raw/customer_data.csv')
df_customer_segment = pd.read_csv(local_datasets + 'raw/customer_segment.csv')

In [9]:
df_customer_data.shape

(21321993, 11)

In [10]:
df_customer_segment.shape

(15604162, 11)

In [11]:
df_customer_data.head(1)

,customerid,city,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides
0,677aa63e96f8353e0483480f,Bangalore,35,32,0,0,0,0,0,0,0


In [12]:
df_customer_data.customerid.nunique()

20372699

In [13]:
df_customer_segment.head(1)

,customer_id,customer_obj_mobile,city_name,customer_id.1,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment
0,65733a6024a0f87d756edfdc,8.823815e+09,Ahmedabad,65733a6024a0f87d756edfdc,MALE,GOLD,CHURN_OTB,PHH,UNKNOWN,BI_WEEKLY,BI_WEEKLY


In [14]:
df_customer_segment.customer_id.nunique()

15604162

In [19]:
df_merge = pd.merge(
                    df_customer_data,
                    df_customer_segment,
                    how = 'inner',
                    left_on = ['customerid', 'city'],
                    right_on = ['customer_id', 'city_name']
                   )

In [20]:
df_merge.head(1)

,customerid,city,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,customer_id,customer_obj_mobile,city_name,customer_id.1,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment
0,5c5297934a267149c7804b96,Chennai,10,10,1,1,0,0,0,0,0,5c5297934a267149c7804b96,9.171172e+09,Chennai,5c5297934a267149c7804b96,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,MONTHLY


In [21]:
df_merge.shape

(15603622, 22)

In [22]:
df_merge.customerid.nunique()

15603622

In [57]:
df_data = df_merge[[
                    'customerid', 'city', 'customer_obj_mobile',
                    'ao_sessions',	'fe_sessions',	'rr_sessions',
                    'link_gross_rides',	'auto_gross_rides',	'cab_gross_rides',	'link_net_rides',	'net_rides',	'cab_net_rides',
                    'gender',	'taxi_retention_segments',	'taxi_lifetime_stage',	'taxi_ltr_segment',	
                    'taxi_service_affinity',	'taxi_regularity_segment', 'taxi_fe_regularity_segment'
                    ]].copy()

In [58]:
df_data.head(5)

,customerid,city,customer_obj_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment
0,5c5297934a267149c7804b96,Chennai,9.171172e+09,10,10,1,1,0,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,MONTHLY
1,60cb106f7c2b2649e0686c64,Bangalore,6.300411e+09,300,233,44,45,1,0,39,1,0,MALE,ELITE,COMMITTED,PHH,NO_AFFINITY,DAILY,DAILY
2,62a5ac0d5a4705a8f5a2c016,Delhi,9.971917e+09,18,19,1,0,1,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,BI_WEEKLY
3,64f0ca98b77f5d2a7c9d7b6a,Delhi,9.278336e+09,29,29,2,0,0,3,0,0,2,MALE,PLATINUM,COMMITTED,PHH,LINK_AUTO,MONTHLY,RARE_NEED
4,5cc4631654bc7263ff52a964,Delhi,6.291313e+09,82,59,14,2,14,4,0,4,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,DAILY,DAILY


In [59]:
df_data['customer_mobile'] = df_data['customer_obj_mobile'].fillna(0).astype(float).astype(int).astype(str)
df_data['is_valid_length'] = df_data['customer_mobile'].str.len() == 10
df_data['is_valid_length'].value_counts()

is_valid_length
True     15603469
False         153
Name: count, dtype: int64

In [60]:
df_data.head(5)

,customerid,city,customer_obj_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,customer_mobile,is_valid_length
0,5c5297934a267149c7804b96,Chennai,9.171172e+09,10,10,1,1,0,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,MONTHLY,9171171919,True
1,60cb106f7c2b2649e0686c64,Bangalore,6.300411e+09,300,233,44,45,1,0,39,1,0,MALE,ELITE,COMMITTED,PHH,NO_AFFINITY,DAILY,DAILY,6300410928,True
2,62a5ac0d5a4705a8f5a2c016,Delhi,9.971917e+09,18,19,1,0,1,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,BI_WEEKLY,9971916894,True
3,64f0ca98b77f5d2a7c9d7b6a,Delhi,9.278336e+09,29,29,2,0,0,3,0,0,2,MALE,PLATINUM,COMMITTED,PHH,LINK_AUTO,MONTHLY,RARE_NEED,9278336392,True
4,5cc4631654bc7263ff52a964,Delhi,6.291313e+09,82,59,14,2,14,4,0,4,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,DAILY,DAILY,6291313175,True


In [63]:
df = df_data[df_data['is_valid_length'] == True][[
                    'customerid', 'city', 'customer_mobile',
                    'ao_sessions',	'fe_sessions',	'rr_sessions',
                    'link_gross_rides',	'auto_gross_rides',	'cab_gross_rides',	'link_net_rides',	'net_rides',	'cab_net_rides',
                    'gender',	'taxi_retention_segments',	'taxi_lifetime_stage',	'taxi_ltr_segment',	
                    'taxi_service_affinity',	'taxi_regularity_segment', 'taxi_fe_regularity_segment'
                    ]]

In [64]:
df.head(5)

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment
0,5c5297934a267149c7804b96,Chennai,9171171919,10,10,1,1,0,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,MONTHLY
1,60cb106f7c2b2649e0686c64,Bangalore,6300410928,300,233,44,45,1,0,39,1,0,MALE,ELITE,COMMITTED,PHH,NO_AFFINITY,DAILY,DAILY
2,62a5ac0d5a4705a8f5a2c016,Delhi,9971916894,18,19,1,0,1,0,0,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,BI_WEEKLY
3,64f0ca98b77f5d2a7c9d7b6a,Delhi,9278336392,29,29,2,0,0,3,0,0,2,MALE,PLATINUM,COMMITTED,PHH,LINK_AUTO,MONTHLY,RARE_NEED
4,5cc4631654bc7263ff52a964,Delhi,6291313175,82,59,14,2,14,4,0,4,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,DAILY,DAILY


In [69]:
print(df.shape)
print(df.customerid.nunique())
print(df.customer_mobile.nunique())

(15603469, 19)
15603469
15603443


In [73]:
result = (
    df\
        .groupby('customer_mobile')['customerid']\
        .nunique()\
        .reset_index(name='cux')
    )

result = result[result['cux'] > 1]
exclude_list = result['customer_mobile'].tolist()

['6264897579',
 '6363027744',
 '7022656534',
 '7200581303',
 '7207608298',
 '7838670577',
 '8008278089',
 '8148962694',
 '8248906232',
 '8284884429',
 '8553793353',
 '8555044803',
 '8838354346',
 '8892880561',
 '8939831323',
 '9035456496',
 '9035497784',
 '9066566336',
 '9108287202',
 '9346644740',
 '9599367028',
 '9688196912',
 '9741801141',
 '9845927674',
 '9898361886',
 '9966510499']

In [75]:
df1 = df[~df['customer_mobile'].isin(exclude_list)]

In [76]:
print(df1.shape)
print(df1.customerid.nunique())
print(df1.customer_mobile.nunique())

(15603417, 19)
15603417
15603417


In [77]:
df1[ df1['customerid'].isin(['5cc4631654bc7263ff52a964'])].T

,4
customerid,5cc4631654bc7263ff52a964
city,Delhi
customer_mobile,6291313175
ao_sessions,82
fe_sessions,59
rr_sessions,14
link_gross_rides,2
auto_gross_rides,14
cab_gross_rides,4
link_net_rides,0


In [78]:
df_stratified_random_sampling = df1.copy()

In [79]:
df_stratified_random_sampling.isnull().sum()

customerid                          0
city                                0
customer_mobile                     0
ao_sessions                         0
fe_sessions                         0
rr_sessions                         0
link_gross_rides                    0
auto_gross_rides                    0
cab_gross_rides                     0
link_net_rides                      0
net_rides                           0
cab_net_rides                       0
gender                        3180938
taxi_retention_segments       3180764
taxi_lifetime_stage           3180764
taxi_ltr_segment              3180764
taxi_service_affinity         3180764
taxi_regularity_segment       3180764
taxi_fe_regularity_segment    3180764
dtype: int64

In [81]:
# Columns to fill null values in
columns_to_fill = ['gender', 'taxi_retention_segments',
                   'taxi_lifetime_stage', 'taxi_ltr_segment', 
                   'taxi_service_affinity', 'taxi_regularity_segment', 'taxi_fe_regularity_segment']

# Fill null values in the specified columns with a specific value (e.g., 0)
df_stratified_random_sampling[columns_to_fill] = df_stratified_random_sampling[columns_to_fill].fillna('NA')

In [82]:
df_stratified_random_sampling.isnull().sum()

customerid                    0
city                          0
customer_mobile               0
ao_sessions                   0
fe_sessions                   0
rr_sessions                   0
link_gross_rides              0
auto_gross_rides              0
cab_gross_rides               0
link_net_rides                0
net_rides                     0
cab_net_rides                 0
gender                        0
taxi_retention_segments       0
taxi_lifetime_stage           0
taxi_ltr_segment              0
taxi_service_affinity         0
taxi_regularity_segment       0
taxi_fe_regularity_segment    0
dtype: int64

In [83]:
df_stratified_random_sampling.info()

<class 'pandas.core.frame.DataFrame'>
Index: 15603417 entries, 0 to 15603621
Data columns (total 19 columns):
 #   Column                      Dtype 
---  ------                      ----- 
 0   customerid                  object
 1   city                        object
 2   customer_mobile             object
 3   ao_sessions                 int64 
 4   fe_sessions                 int64 
 5   rr_sessions                 int64 
 6   link_gross_rides            int64 
 7   auto_gross_rides            int64 
 8   cab_gross_rides             int64 
 9   link_net_rides              int64 
 10  net_rides                   int64 
 11  cab_net_rides               int64 
 12  gender                      object
 13  taxi_retention_segments     object
 14  taxi_lifetime_stage         object
 15  taxi_ltr_segment            object
 16  taxi_service_affinity       object
 17  taxi_regularity_segment     object
 18  taxi_fe_regularity_segment  object
dtypes: int64(9), object(10)
memory usage: 2.3+ GB

## Stratified Random Sampling

In [84]:
df_stratified_random_sampling.shape

(15603417, 19)

In [85]:
df_stratified_random_sampling.customerid.nunique()

15603417

In [86]:
df_stratified_random_sampling.columns

Index(['customerid', 'city', 'customer_mobile', 'ao_sessions', 'fe_sessions',
       'rr_sessions', 'link_gross_rides', 'auto_gross_rides',
       'cab_gross_rides', 'link_net_rides', 'net_rides', 'cab_net_rides',
       'gender', 'taxi_retention_segments', 'taxi_lifetime_stage',
       'taxi_ltr_segment', 'taxi_service_affinity', 'taxi_regularity_segment',
       'taxi_fe_regularity_segment'],
      dtype='object')

In [87]:
stratify_grouper_col =['city', 'ao_sessions', 'fe_sessions',
                       'rr_sessions', 'link_gross_rides', 'auto_gross_rides',
                       'cab_gross_rides', 'link_net_rides', 'net_rides', 'cab_net_rides',
                       'gender', 'taxi_retention_segments', 'taxi_lifetime_stage',
                       'taxi_ltr_segment', 'taxi_service_affinity', 'taxi_regularity_segment',
                       'taxi_fe_regularity_segment'
                      ]

In [92]:
random_seed = 42

# Split the df_stratified_random_sampling into two parts for initial 'control' and 'temp'
df_train_control, df_test = train_test_split(df_stratified_random_sampling, 
                                             test_size=1/2, 
                                             random_state=random_seed
                                             # stratify=df_stratified_random_sampling[stratify_grouper_col]
                                            )

In [93]:

df_train_control = df_train_control.sample(min(500000, len(df_train_control)), random_state=random_seed)
df_test = df_test.sample(min(500000, len(df_test)), random_state=random_seed)

# Assign group labels
df_train_control['group_tc'] = 'control'
df_test['group_tc'] = 'test'

df_final_sample = pd.concat([df_train_control, df_test])

In [94]:
print(df_stratified_random_sampling.shape)
print(df_final_sample.shape)

(15603417, 19)
(1000000, 20)


In [95]:
df_final_sample

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc
3716388,6705e6eed10b4ab28fd00d85,Delhi,6207413061,39,34,4,5,0,0,3,0,0,MALE,DORMANT,DORMANT,HH,UNKNOWN,UNKNOWN,BI_WEEKLY,control
7344337,5db8ca22c54ee121252b7b72,Hyderabad,6300802372,38,28,4,3,1,0,3,1,0,FEMALE,GOLD,SUSTENANCE,PHH,ONLY_LINK,WEEKLY,WEEKLY,control
853766,646dfe30d87110491d6df3d7,Vijayawada,9966007979,35,25,5,4,1,0,4,1,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,WEEKLY,WEEKLY,control
6519994,66dc2b682ae68b2390e2ecfb,Pune,9766135494,0,0,0,0,1,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,control
10504843,612f9d6acb05bc1cbf2df219,Hyderabad,8790673944,20,8,2,0,2,0,0,1,0,UNKNOWN,INACTIVE,INACTIVE,HH,UNKNOWN,UNKNOWN,UNKNOWN,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5133999,5e5d144001c1d592e3743939,Hyderabad,6301655004,20,20,4,4,0,0,4,0,0,FEMALE,INACTIVE,INACTIVE,PHH,UNKNOWN,RARE_NEED,UNKNOWN,test
5526059,5fbd50c17e483f5ff3fe5af6,Bangalore,8971400959,130,100,17,20,0,0,13,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,MONTHLY,MONTHLY,test
5581249,5f1c0ae9ea5a8b2edc789205,Delhi,9650135370,9,9,2,1,1,1,0,1,0,MALE,SILVER,HOOK,PHH,UNKNOWN,MONTHLY,MONTHLY,test
14654977,629d77f5737cf7f02283e56d,Bangalore,7349771897,0,0,0,0,2,0,0,2,0,NA,NA,NA,NA,NA,NA,NA,test


In [96]:
df_final_sample['customer_mobile'].nunique()

1000000

### Balance Groups

In [97]:
df_final_sample.columns

Index(['customerid', 'city', 'customer_mobile', 'ao_sessions', 'fe_sessions',
       'rr_sessions', 'link_gross_rides', 'auto_gross_rides',
       'cab_gross_rides', 'link_net_rides', 'net_rides', 'cab_net_rides',
       'gender', 'taxi_retention_segments', 'taxi_lifetime_stage',
       'taxi_ltr_segment', 'taxi_service_affinity', 'taxi_regularity_segment',
       'taxi_fe_regularity_segment', 'group_tc'],
      dtype='object')

In [98]:
df_final_sample.head(5)

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc
3716388,6705e6eed10b4ab28fd00d85,Delhi,6207413061,39,34,4,5,0,0,3,0,0,MALE,DORMANT,DORMANT,HH,UNKNOWN,UNKNOWN,BI_WEEKLY,control
7344337,5db8ca22c54ee121252b7b72,Hyderabad,6300802372,38,28,4,3,1,0,3,1,0,FEMALE,GOLD,SUSTENANCE,PHH,ONLY_LINK,WEEKLY,WEEKLY,control
853766,646dfe30d87110491d6df3d7,Vijayawada,9966007979,35,25,5,4,1,0,4,1,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,WEEKLY,WEEKLY,control
6519994,66dc2b682ae68b2390e2ecfb,Pune,9766135494,0,0,0,0,1,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,control
10504843,612f9d6acb05bc1cbf2df219,Hyderabad,8790673944,20,8,2,0,2,0,0,1,0,UNKNOWN,INACTIVE,INACTIVE,HH,UNKNOWN,UNKNOWN,UNKNOWN,control


In [100]:
df_final_sample.groupby(['group_tc'])\
.agg(user_base = pd.NamedAgg('customerid', 'nunique'),
     customer_mobile = pd.NamedAgg('customer_mobile', 'nunique'),
     city_distr = pd.NamedAgg(column='city', 
                              aggfunc=lambda x: x.value_counts().to_dict()),
     ao_sessions = pd.NamedAgg('ao_sessions', 'sum'),
     fe_sessions = pd.NamedAgg('fe_sessions', 'sum'),
     rr_sessions = pd.NamedAgg('rr_sessions', 'sum'),
     link_gross_rides = pd.NamedAgg('link_gross_rides', 'sum'),
     auto_gross_rides = pd.NamedAgg('auto_gross_rides', 'sum'),
     cab_gross_rides = pd.NamedAgg('cab_gross_rides', 'sum'),
     link_net_rides = pd.NamedAgg('link_net_rides', 'sum'),
     auto_net_rides = pd.NamedAgg('net_rides', 'sum'),
     cab_net_rides = pd.NamedAgg('cab_net_rides', 'sum'),

     retention_segment_dist = pd.NamedAgg(column='taxi_retention_segments', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     lifetime_stage_dist = pd.NamedAgg(column='taxi_lifetime_stage', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     ltr_segment_dist = pd.NamedAgg(column='taxi_ltr_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     service_affinity_dist = pd.NamedAgg(column='taxi_service_affinity', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     regularity_segment_dist = pd.NamedAgg(column='taxi_regularity_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     fe_regularity_segment_dist = pd.NamedAgg(column='taxi_fe_regularity_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
        
    ).T

group_tc,control,test
user_base,500000,500000
customer_mobile,500000,500000
city_distr,"{'Hyderabad': 124499, 'Delhi': 109708, 'Bangalore': 91696, 'Chennai': 60773, 'Jaipur': 23084, 'Kolkata': 22697, 'Ahmedabad': 20282, 'Mumbai': 16871, 'Pune': 16542, 'Vijayawada': 13848}","{'Hyderabad': 125522, 'Delhi': 109390, 'Bangalore': 91300, 'Chennai': 60957, 'Jaipur': 22859, 'Kolkata': 22809, 'Ahmedabad': 20004, 'Mumbai': 17144, 'Pune': 16633, 'Vijayawada': 13382}"
ao_sessions,21077871,21089428
fe_sessions,15332761,15319336
rr_sessions,2315338,2314265
link_gross_rides,1116000,1105899
auto_gross_rides,1223260,1225935
cab_gross_rides,476252,478008
link_net_rides,767404,763205


In [102]:
df_final_sample.groupby(['city', 'group_tc'])\
.agg(user_base = pd.NamedAgg('customerid', 'nunique'),
     customer_mobile = pd.NamedAgg('customer_mobile', 'nunique'),
     ao_sessions = pd.NamedAgg('ao_sessions', 'sum'),
     fe_sessions = pd.NamedAgg('fe_sessions', 'sum'),
     rr_sessions = pd.NamedAgg('rr_sessions', 'sum'),
     link_gross_rides = pd.NamedAgg('link_gross_rides', 'sum'),
     auto_gross_rides = pd.NamedAgg('auto_gross_rides', 'sum'),
     cab_gross_rides = pd.NamedAgg('cab_gross_rides', 'sum'),
     link_net_rides = pd.NamedAgg('link_net_rides', 'sum'),
     auto_net_rides = pd.NamedAgg('net_rides', 'sum'),
     cab_net_rides = pd.NamedAgg('cab_net_rides', 'sum'),

     retention_segment_dist = pd.NamedAgg(column='taxi_retention_segments', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     lifetime_stage_dist = pd.NamedAgg(column='taxi_lifetime_stage', 
                                        aggfunc=lambda x: x.value_counts().to_dict()), 
     
     ltr_segment_dist = pd.NamedAgg(column='taxi_ltr_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     service_affinity_dist = pd.NamedAgg(column='taxi_service_affinity', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     regularity_segment_dist = pd.NamedAgg(column='taxi_regularity_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),

     fe_regularity_segment_dist = pd.NamedAgg(column='taxi_fe_regularity_segment', 
                                         aggfunc=lambda x: x.value_counts().to_dict()),
     
     gender_distr = pd.NamedAgg(column='gender', 
                                aggfunc=lambda x: x.value_counts().to_dict()),
        
    )

user_base  customer_mobile  ao_sessions  fe_sessions  \
city       group_tc                                                         
Ahmedabad  control       20282            20282       673320       523458   
           test          20004            20004       664349       515314   
Bangalore  control       91696            91696      4575974      3369780   
           test          91300            91300      4562715      3359770   
Chennai    control       60773            60773      2517582      1828664   
           test          60957            60957      2531325      1838290   
Delhi      control      109708           109708      5480849      3994260   
           test         109390           109390      5514524      4007805   
Hyderabad  control      124499           124499      5180569      3605479   
           test         125522           125522      5223822      3631971   
Jaipur     control       23084            23084       830943       617700   
           test          22859            22859       800844       595193   
Kolkata    control       22697            22697       559187       416803   
           test          22809            22809       555520       413864   
Mumbai     control       16871            16871       383935       287063   
           test          17144            17144       383658       288091   
Pune       control       16542            16542       414376       322419   
           test          16633            16633       407551       316628   
Vijayawada control       13848            13848       461136       367135   
           test          13382            13382       445120       352410   

                     rr_sessions  link_gross_rides  auto_gross_rides  \
city       group_tc                                                    
Ahmedabad  control         82420             49763             33138   
           test            81041             47890             34092   
Bangalore  control        468431            178021            319239   
           test           469004            176573            322257   
Chennai    control        274503            131516            165465   
           test           276745            131960            168314   
Delhi      control        555419            321113            179974   
           test           556814            318350            181169   
Hyderabad  control        560600            266213            340215   
           test           562952            265479            339472   
Jaipur     control         91958             47113             48898   
           test            88131             44603             45821   
Kolkata    control         85595             80045                 0   
           test            85237             79644                 0   
Mumbai     control         65178                 0             46945   
           test            66165                 0             48166   
Pune       control         73522                 0             60567   
           test            72619                 0             59680   
Vijayawada control         57712             42216             28819   
           test            55557             41400             26964   

                     cab_gross_rides  link_net_rides  auto_net_rides  \
city       group_tc                                                    
Ahmedabad  control             21430           30835           21181   
           test                20542           30027           21558   
Bangalore  control             72677          109675          223580   
           test                71956          109024          225573   
Chennai    control             31871           90669          123451   
           test                31976           90737          125825   
Delhi      control            173188          220179           97965   
           test               175159          218844           98822   
Hyderabad  control   

In [103]:
df_final_sample.reset_index(drop=True)

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc
0,6705e6eed10b4ab28fd00d85,Delhi,6207413061,39,34,4,5,0,0,3,0,0,MALE,DORMANT,DORMANT,HH,UNKNOWN,UNKNOWN,BI_WEEKLY,control
1,5db8ca22c54ee121252b7b72,Hyderabad,6300802372,38,28,4,3,1,0,3,1,0,FEMALE,GOLD,SUSTENANCE,PHH,ONLY_LINK,WEEKLY,WEEKLY,control
2,646dfe30d87110491d6df3d7,Vijayawada,9966007979,35,25,5,4,1,0,4,1,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,WEEKLY,WEEKLY,control
3,66dc2b682ae68b2390e2ecfb,Pune,9766135494,0,0,0,0,1,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,control
4,612f9d6acb05bc1cbf2df219,Hyderabad,8790673944,20,8,2,0,2,0,0,1,0,UNKNOWN,INACTIVE,INACTIVE,HH,UNKNOWN,UNKNOWN,UNKNOWN,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,5e5d144001c1d592e3743939,Hyderabad,6301655004,20,20,4,4,0,0,4,0,0,FEMALE,INACTIVE,INACTIVE,PHH,UNKNOWN,RARE_NEED,UNKNOWN,test
999996,5fbd50c17e483f5ff3fe5af6,Bangalore,8971400959,130,100,17,20,0,0,13,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,MONTHLY,MONTHLY,test
999997,5f1c0ae9ea5a8b2edc789205,Delhi,9650135370,9,9,2,1,1,1,0,1,0,MALE,SILVER,HOOK,PHH,UNKNOWN,MONTHLY,MONTHLY,test
999998,629d77f5737cf7f02283e56d,Bangalore,7349771897,0,0,0,0,2,0,0,2,0,NA,NA,NA,NA,NA,NA,NA,test


In [104]:
##Local extract
df_final_sample.to_csv(local_datasets + 'processed/data.csv', index=False)
# df_final_sample = pd.read_csv(local_datasets + 'processed/data.csv')

In [106]:
# Check for common values in 'user_id' among the three DataFrames
common_values_df1_df2 = df_train_control[df_train_control['customerid'].isin(df_test['customerid'])]['customerid'].tolist()
common_values_df1_df2

[]

In [108]:
print(df_train_control.shape)
print(df_test.shape)

print(df_train_control.customerid.nunique())
print(df_test.customerid.nunique())


(500000, 20)
(500000, 20)
500000
500000


In [109]:
df_train_control[df_train_control['customerid'].isin(df_test['customerid'])]

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc


In [110]:
df_test[df_test['customerid'].isin(df_train_control['customerid'])]

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc


## User Selector for Experiment

In [111]:
df_train_control.reset_index(drop=True)

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc
0,6705e6eed10b4ab28fd00d85,Delhi,6207413061,39,34,4,5,0,0,3,0,0,MALE,DORMANT,DORMANT,HH,UNKNOWN,UNKNOWN,BI_WEEKLY,control
1,5db8ca22c54ee121252b7b72,Hyderabad,6300802372,38,28,4,3,1,0,3,1,0,FEMALE,GOLD,SUSTENANCE,PHH,ONLY_LINK,WEEKLY,WEEKLY,control
2,646dfe30d87110491d6df3d7,Vijayawada,9966007979,35,25,5,4,1,0,4,1,0,MALE,ELITE,COMMITTED,PHH,UNKNOWN,WEEKLY,WEEKLY,control
3,66dc2b682ae68b2390e2ecfb,Pune,9766135494,0,0,0,0,1,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,control
4,612f9d6acb05bc1cbf2df219,Hyderabad,8790673944,20,8,2,0,2,0,0,1,0,UNKNOWN,INACTIVE,INACTIVE,HH,UNKNOWN,UNKNOWN,UNKNOWN,control
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,665490e11c9f5b372e1147a8,Delhi,8368323637,5,14,1,0,2,0,0,0,0,MALE,GOLD,SUSTENANCE,PHH,ONLY_LINK,WEEKLY,BI_WEEKLY,control
499996,670a124a476051ece53b630c,Delhi,7703971722,15,15,2,0,1,2,0,0,1,FEMALE,HH,HANDHOLDING,HH,UNKNOWN,UNKNOWN,RARE_NEED,control
499997,5d0d05cac2a56b449f9e3310,Hyderabad,7879115445,36,37,2,0,1,1,0,1,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,MONTHLY,MONTHLY,control
499998,6531ded739ce6b6f2da382c6,Chennai,9043331433,4,4,1,2,0,0,0,0,0,MALE,GOLD,SUSTENANCE,PHH,UNKNOWN,BI_WEEKLY,BI_WEEKLY,control


In [112]:
df_test.reset_index(drop=True)

,customerid,city,customer_mobile,ao_sessions,fe_sessions,rr_sessions,link_gross_rides,auto_gross_rides,cab_gross_rides,link_net_rides,net_rides,cab_net_rides,gender,taxi_retention_segments,taxi_lifetime_stage,taxi_ltr_segment,taxi_service_affinity,taxi_regularity_segment,taxi_fe_regularity_segment,group_tc
0,65f01b8502500313d302a12a,Delhi,8348821149,15,11,2,0,3,0,0,1,0,NA,NA,NA,NA,NA,NA,NA,test
1,66de724ef76fe845d6d4f48a,Delhi,9453522807,21,21,2,1,1,0,1,0,0,MALE,SILVER,HOOK,PHH,ONLY_CAB,RARE_NEED,RARE_NEED,test
2,60af1d42c41d740c796fff8f,Hyderabad,9032559753,18,17,2,2,0,0,2,0,0,MALE,GOLD,HOOK,PHH,UNKNOWN,BI_WEEKLY,WEEKLY,test
3,6448b2f786dd2131d33ac350,Hyderabad,8073509872,17,17,3,2,2,0,2,1,0,MALE,GOLD,CHURN_OTB,PHH,ONLY_AUTO,WEEKLY,BI_WEEKLY,test
4,5e103962a10a71228f8d2d5e,Jaipur,7024843628,45,36,8,8,0,1,4,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,RARE_NEED,UNKNOWN,test
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499995,5e5d144001c1d592e3743939,Hyderabad,6301655004,20,20,4,4,0,0,4,0,0,FEMALE,INACTIVE,INACTIVE,PHH,UNKNOWN,RARE_NEED,UNKNOWN,test
499996,5fbd50c17e483f5ff3fe5af6,Bangalore,8971400959,130,100,17,20,0,0,13,0,0,MALE,DORMANT,DORMANT,PHH,UNKNOWN,MONTHLY,MONTHLY,test
499997,5f1c0ae9ea5a8b2edc789205,Delhi,9650135370,9,9,2,1,1,1,0,1,0,MALE,SILVER,HOOK,PHH,UNKNOWN,MONTHLY,MONTHLY,test
499998,629d77f5737cf7f02283e56d,Bangalore,7349771897,0,0,0,0,2,0,0,2,0,NA,NA,NA,NA,NA,NA,NA,test


In [115]:
df_train_control['customer_mobile'] = df_train_control['customer_mobile'].astype(float).astype(int)

In [116]:
df_test['customer_mobile'] = df_test['customer_mobile'].astype(float).astype(int)

In [117]:
df_train_control['customer_mobile']

3716388     6207413061
7344337     6300802372
853766      9966007979
6519994     9766135494
10504843    8790673944
               ...    
6123665     8368323637
4585547     7703971722
4865414     7879115445
5505476     9043331433
9640581     8369018935
Name: customer_mobile, Length: 500000, dtype: int64

In [118]:
df_test['customer_mobile']

8735817     8348821149
2585278     9453522807
6623667     9032559753
10225742    8073509872
9757232     7024843628
               ...    
5133999     6301655004
5526059     8971400959
5581249     9650135370
14654977    7349771897
8754211     7483452364
Name: customer_mobile, Length: 500000, dtype: int64

In [121]:
def split_fn(df, type):

    chunks = np.array_split(df, 5)

    folder = type
    for i, chunk in enumerate(chunks):
        chunk.to_csv(f'/Users/E2074/local-datasets/customer/ads-monetisation/jackport/processed/{folder}_user_selector_{i + 1}.csv', 
                     header=False,
                     index=False
                    )
    
    return print('done')

In [122]:
split_fn(df_train_control[['customer_mobile']], 'control')

done


In [123]:
split_fn(df_test[['customer_mobile']], 'test')

done


In [124]:
df_train_control[['customer_mobile']]

,customer_mobile
3716388,6207413061
7344337,6300802372
853766,9966007979
6519994,9766135494
10504843,8790673944
...,...
6123665,8368323637
4585547,7703971722
4865414,7879115445
5505476,9043331433


In [125]:
df_test[['customer_mobile']]

,customer_mobile
8735817,8348821149
2585278,9453522807
6623667,9032559753
10225742,8073509872
9757232,7024843628
...,...
5133999,6301655004
5526059,8971400959
5581249,9650135370
14654977,7349771897


In [ ]:
# Current - 5L vs 5L (mix of 10 cities)

In [ ]:
# Current - 5L vs 5L (mix of 10 cities)